In [2]:
import re
import numpy as np
import pandas as pd

df = pd.read_csv("8891_motorcycles.csv")
df.columns = df.columns.str.strip()

COLS = {
    "category": "category",
    "title": "title",
    "brand": "brand",
    "model": "model",
    "price": "price_twd",
    "raw_price": "raw_price",
    "year": "year",
    "cc": "gas_displacement",
    "region": "region",
    "url": "url",
}

CURRENT_YEAR = 2026
PLACEHOLDER_PRICES = {1111, 1234, 8888, 9999, 12345, 88888, 99999, 123456, 888888, 999999}


def clean_price(val) -> float:
    if pd.isna(val):
        return np.nan
    s = str(val).replace(",", "").replace(" ", "").strip()
    match = re.search(r"(\d+(?:\.\d+)?)\s*(萬)?", s)
    if not match:
        return np.nan

    num = float(match.group(1))
    if match.group(2) == "萬":
        num *= 10_000

    price = int(round(num))
    if price < 5_000 or price > 3_500_000 or price in PLACEHOLDER_PRICES:
        return np.nan
    return float(price)


def clean_year(val) -> float:
    if pd.isna(val):
        return np.nan
    match = re.search(r"(\d{2,4})", str(val))
    if not match:
        return np.nan

    yr = int(match.group(1))
    if 70 <= yr <= 116:  # ROC (民國) year -> Gregorian
        yr += 1911

    if 1980 <= yr <= CURRENT_YEAR + 1:
        return float(yr)
    return np.nan


def extract_mileage_from_title(title: str) -> float:
    """Extract mileage when stated in title (e.g., '實跑3036公里', '實跑6.2萬', '只跑2千')."""
    if pd.isna(title):
        return np.nan
    s = str(title).replace(",", "").strip()
    match = re.search(r"(?:實跑|里程|僅跑|只跑)\s*(\d+(?:\.\d+)?)\s*(萬|千)?\s*(?:公里|km)?", s, flags=re.IGNORECASE)
    if not match:
        return np.nan

    num = float(match.group(1))
    unit = match.group(2)
    if unit == "萬":
        num *= 10_000
    elif unit == "千":
        num *= 1_000

    km = int(round(num))
    if 0 <= km <= 300_000:
        return float(km)
    return np.nan


def extract_cc(cc_val, title: str, brand: str) -> float:
    if brand == "GOGORO":
        return 0.0

    # 1. Parse gas_displacement (handles both '125cc'/'155' and liter format like '0.2L'/'0.6L')
    if pd.notna(cc_val):
        s_cc = str(cc_val).strip()
        m_liter = re.search(r"(\d+(?:\.\d+)?)\s*l\b", s_cc, flags=re.IGNORECASE)
        if m_liter:
            cc_num = round(float(m_liter.group(1)) * 1000)
            if 49 <= cc_num <= 2500:
                return float(cc_num)

        m_num = re.search(r"(\d{2,4})", s_cc)
        if m_num and 49 <= int(m_num.group(1)) <= 2500:
            return float(m_num.group(1))

    # 2. Fallback to title
    if pd.notna(title):
        m_title = re.search(r"\b(\d{2,4})\s*(?:cc|c\.c\.)\b", str(title), flags=re.IGNORECASE)
        if m_title and 49 <= int(m_title.group(1)) <= 2500:
            return float(m_title.group(1))

    return np.nan

In [3]:
BRAND_MAP = {
    "SYM": ["sym", "三陽"],
    "KYMCO": ["kymco", "光陽"],
    "YAMAHA": ["yamaha", "山葉"],
    "GOGORO": ["gogoro", "睿能"],
    "PGO": ["pgo", "摩特動力", "比雅久"],
    "AEON": ["aeon", "宏佳騰"],
    "SUZUKI": ["suzuki", "台鈴", "鈴木"],
    "HONDA": ["honda", "本田"],
    "KAWASAKI": ["kawasaki", "川崎"],
    "VESPA": ["vespa", "偉士牌"],
    "DUCATI": ["ducati", "杜卡迪"],
    "BMW": ["bmw"],
}

MODEL_PATTERNS = [
    (r"jet\s*sl\+?", "JET SL"),
    (r"jet\s*sr", "JET SR"),
    (r"jet\s*s\b", "JET S"),
    (r"\bdrg\b", "DRG"),
    (r"\bmmbcu\b|曼巴", "MMBCU"),
    (r"\bkrn\b", "KRN"),
    (r"\bfiddle\b", "FIDDLE"),
    (r"\bclbcu\b", "CLBCU"),
    (r"勁戰|cygnus", "CYGNUS"),
    (r"\bforce\b", "FORCE"),
    (r"\bsmax\b", "SMAX"),
    (r"\bbws\b", "BWS"),
    (r"racing\s*s|雷霆s", "RACING S"),
    (r"\bkrv\b", "KRV"),
    (r"\bmany\b", "MANY"),
    (r"\bvjr\b", "VJR"),
    (r"\bgp\s*125\b|\bgp\b", "GP"),
    (r"\btigra\b", "TIGRA"),
    (r"\bmt[-\s]?03\b", "MT-03"),
    (r"\bmt[-\s]?07\b", "MT-07"),
    (r"\bmt[-\s]?09\b", "MT-09"),
    (r"\br15\b", "R15"),
    (r"\br3\b", "R3"),
    (r"\br7\b", "R7"),
    (r"cbr\s*150r?", "CBR150R"),
    (r"cbr\s*500r?", "CBR500R"),
    (r"cbr\s*650r?", "CBR650R"),
    (r"cb\s*300r?", "CB300R"),
    (r"cb\s*650r?", "CB650R"),
    (r"ninja\s*400", "NINJA 400"),
    (r"\bz400\b", "Z400"),
    (r"viva\s*mix", "VIVA MIX"),
    (r"gogoro\s*2", "GOGORO 2"),
    (r"gogoro\s*3", "GOGORO 3"),
]


def standardize_brand(text: str) -> str:
    if pd.isna(text):
        return "OTHER"
    s = str(text).lower()
    for canonical, aliases in BRAND_MAP.items():
        if any(alias in s for alias in aliases):
            return canonical
    return "OTHER"


def standardize_model(model_col_val: str, title: str) -> str:
    combined = f"{'' if pd.isna(model_col_val) else model_col_val} {'' if pd.isna(title) else title}".lower()
    for pattern, canonical_model in MODEL_PATTERNS:
        if re.search(pattern, combined):
            return canonical_model
    # Fallback to cleaned scraped model column if non-empty
    if pd.notna(model_col_val) and str(model_col_val).strip():
        return str(model_col_val).strip().upper()
    return "OTHER"

In [4]:
clean_df = df.copy()

# Deduplicate by listing URL first
clean_df = clean_df.drop_duplicates(subset=[COLS["url"]]).reset_index(drop=True)

# Price: parse price_twd first, coalesce missing values from raw_price
price_primary = clean_df[COLS["price"]].apply(clean_price)
price_fallback = clean_df[COLS["raw_price"]].apply(clean_price)
clean_df["price_ntd"] = price_primary.fillna(price_fallback)

# Year, age & optional title-extracted mileage
clean_df["model_year"] = clean_df[COLS["year"]].apply(clean_year)
clean_df["vehicle_age"] = CURRENT_YEAR - clean_df["model_year"]
clean_df["mileage_km"] = clean_df[COLS["title"]].apply(extract_mileage_from_title)

zero_km_mask = (clean_df["mileage_km"] == 0) & (clean_df["vehicle_age"] >= 1)
clean_df.loc[zero_km_mask, "mileage_km"] = np.nan

# Brand & model standardization using both dedicated columns and title
brand_combined = clean_df[COLS["brand"]].fillna("") + " " + clean_df[COLS["title"]].fillna("")
clean_df["brand_clean"] = brand_combined.apply(standardize_brand)
clean_df["model_clean"] = [
    standardize_model(m, t) for m, t in zip(clean_df[COLS["model"]], clean_df[COLS["title"]])
]

# Engine displacement
clean_df["cc_clean"] = [
    extract_cc(cc_v, title_v, brand_v)
    for cc_v, title_v, brand_v in zip(clean_df[COLS["cc"]], clean_df[COLS["title"]], clean_df["brand_clean"])
]

# Clean region whitespace
clean_df["region_clean"] = clean_df[COLS["region"]].astype(str).str.strip().replace({"nan": np.nan, "": np.nan})

# Drop rows without valid target price
clean_df = clean_df.dropna(subset=["price_ntd"]).reset_index(drop=True)

# Audit summary
print(f"Remaining valid rows: {len(clean_df)} / {len(df)}")
print("\nMissing values after regex cleaning:")
audit_cols = ["price_ntd", "model_year", "vehicle_age", "brand_clean", "model_clean", "cc_clean", "region_clean", "mileage_km"]
print(clean_df[audit_cols].isna().sum())
print("\nTop brands:\n", clean_df["brand_clean"].value_counts().head(10))
print("\nTop models:\n", clean_df["model_clean"].value_counts().head(15))

Remaining valid rows: 1 / 1500

Missing values after regex cleaning:
price_ntd       0
model_year      0
vehicle_age     0
brand_clean     0
model_clean     0
cc_clean        0
region_clean    0
mileage_km      1
dtype: int64

Top brands:
 brand_clean
SYM    1
Name: count, dtype: int64

Top models:
 model_clean
WOO 115    1
Name: count, dtype: int64


In [5]:
# 1. Export to Parquet (fast loading, preserves exact column data types for ML)
clean_df.to_parquet("8891_motorcycles_clean.parquet", index=False)

# 2. Export to CSV (utf-8-sig prevents Traditional Chinese characters from scrambling in Excel/Numbers)
clean_df.to_csv("8891_motorcycles_clean.csv", index=False, encoding="utf-8-sig")

In [ ]:
%pip install pyarrow